<div align="center">
  <img src="https://raw.githubusercontent.com/eightmm/FoldJAX/main/docs/banner-light.png" width="760" alt="FoldJAX">
  <h2>One input, multiple JAX structure models</h2>
  <p>Build one protein, nucleic-acid, or ligand job, run several FoldJAX models, and compare every result.</p>
</div>

### How to run

1. Use the Colab accelerator runtime you want. FoldJAX detects the active runtime and selects its matching JAX stack and kernels automatically.
2. Choose the models to prepare in the first form, then select **Runtime → Run all**.
3. The dependency cell restarts the runtime once; reconnect and select **Run all** again. Later reruns reuse the matching installation and every verified checkpoint.
4. Setup finishes before the biomolecule form and prediction cells below. The default job contains **protein + RNA + ATP ligand** and runs **Protenix + OpenDDE + Boltz-2 + OpenFold3** sequentially.

| Model | Accepted form input | Checkpoint setup | Checkpoint terms/licence |
|---|---|---|---|
| Protenix | protein, DNA, RNA, CCD/SMILES ligands | public managed download | [Apache-2.0](https://github.com/bytedance/Protenix) |
| OpenDDE | protein, DNA, RNA, CCD/SMILES ligands | public managed download | [see publisher model card](https://huggingface.co/aurekaresearch/OpenDDE) |
| Boltz-2 | protein, DNA, RNA, CCD/SMILES ligands | public managed download | [MIT](https://github.com/jwohlwend/boltz) |
| ESMFold2 | protein, DNA, RNA, ligands, modifications, covalent bonds | released 26.77 GB; requires a runtime reporting more than 16 GiB | [structure weights MIT; ESMC-6B MIT plus its terms](https://huggingface.co/biohub/ESMFold2) |
| OpenFold3 | protein, DNA, RNA, CCD/SMILES ligands | public managed p1, about 2.29 GB | [Apache-2.0 for code and weights](https://github.com/aqlaboratory/openfold-3) |
| AlphaFold 3 | protein, DNA, RNA, CCD/SMILES ligands | manually obtained parameter directory | [AlphaFold 3 parameter terms; not redistributable](https://github.com/google-deepmind/alphafold3) |

> **Accelerator routing:** The notebook follows the active Colab runtime automatically, verifies that JAX sees the selected accelerator, and stops instead of silently falling back to CPU.

> **Checkpoint terms:** The table summarizes the publisher metadata registered by FoldJAX; it is not legal advice. Review the linked publisher/source shown again before each download. FoldJAX does not redistribute restricted parameters.

> **Scientific and privacy scope:** Default mode keeps each model's native sampling settings. Fast demo uses 1 sample, 20 diffusion steps, and 1 recycle; it is a smoke/iteration schedule, not any model's released schedule. Custom schedules, MSA depth caps, and trunk-only execution deliberately change the computation and are recorded in each request. Native confidence fields are shown side by side but are not automatically ranked because names and calibration differ by model. MSA policy auto is the default and sends protein sequences to the public ColabFold MMseqs2 service; select none to avoid an additional MSA service. Colab, Google Drive, uploaded jobs, saved notebook outputs, and downloaded archives can still retain your input.


In [ ]:
# @title 1. Choose models
from html import escape

from IPython.display import HTML, display


def foldjax_card(title, message, *, tone="info", metrics=()):
    palette = {
        "info": ("#2563eb", "#eff6ff"),
        "success": ("#059669", "#ecfdf5"),
        "warning": ("#d97706", "#fffbeb"),
        "error": ("#dc2626", "#fef2f2"),
        "neutral": ("#64748b", "#f8fafc"),
    }
    accent, background = palette[tone]
    chips = "".join(
        f"<span style='display:inline-block;background:white;border:1px solid "
        f"#dbe2ea;border-radius:999px;padding:5px 10px;margin:8px 6px 0 0;"
        f"font-size:12px;color:#334155'><b>{escape(str(label))}</b> "
        f"{escape(str(value))}</span>"
        for label, value in metrics
    )
    display(HTML(
        f"<div style='border:1px solid #dbe2ea;border-left:5px solid {accent};"
        f"border-radius:14px;padding:16px 18px;margin:12px 0;background:{background};"
        f"box-shadow:0 2px 8px rgba(15,23,42,.06)'>"
        f"<div style='font-size:16px;font-weight:700;color:#0f172a'>"
        f"{escape(str(title))}</div>"
        f"<div style='margin-top:5px;color:#475569;line-height:1.55'>"
        f"{escape(str(message))}</div><div>{chips}</div></div>"
    ))


def foldjax_table(frame, title, message=""):
    foldjax_card(title, message, tone="neutral")
    table_html = frame.to_html(index=False, escape=True, border=0)
    display(HTML(
        "<style>"
        ".foldjax-table table{border-collapse:separate;border-spacing:0;"
        "width:100%;font-size:13px}.foldjax-table th{background:#0f172a;"
        "color:white;text-align:left;padding:9px 10px;position:sticky;top:0}"
        ".foldjax-table td{padding:8px 10px;border-bottom:1px solid #e2e8f0;"
        "vertical-align:top}.foldjax-table tr:nth-child(even){background:#f8fafc}"
        "</style><div class='foldjax-table' style='overflow:auto;"
        "border:1px solid #e2e8f0;border-radius:12px;margin:0 0 18px'>"
        f"{table_html}</div>"
    ))

# @markdown **Models — check any compatible combination**
RUN_PROTENIX = True  # @param {type:"boolean"}
RUN_OPENDDE = True  # @param {type:"boolean"}
RUN_BOLTZ2 = True  # @param {type:"boolean"}
RUN_ESMFOLD2 = False  # @param {type:"boolean"}
RUN_OPENFOLD3 = True  # @param {type:"boolean"}
RUN_ALPHAFOLD3 = False  # @param {type:"boolean"}

SELECTED_MODELS = tuple(
    model_name
    for model_name, enabled in (
        ("protenix", RUN_PROTENIX),
        ("opendde", RUN_OPENDDE),
        ("boltz2", RUN_BOLTZ2),
        ("esmfold2", RUN_ESMFOLD2),
        ("openfold3", RUN_OPENFOLD3),
        ("alphafold3", RUN_ALPHAFOLD3),
    )
    if enabled
)
if not SELECTED_MODELS:
    raise ValueError("Select at least one model checkbox.")
foldjax_card(
    "Model choices ready",
    "Run all now installs the selected model stack and prepares its weights.",
    tone="success",
    metrics=(("models", ", ".join(SELECTED_MODELS)),),
)


In [ ]:
# @title 2. Detect the accelerator and install its pinned FoldJAX stack
import os
import shutil
import signal
import subprocess
import sys
from pathlib import Path

FOLDJAX_REF = "52f8c8890956acedac5e293a6bddc2e8b5449ba4"
JAX_VERSION = "0.11.1"

if sys.version_info[:2] != (3, 13):
    raise RuntimeError(
        f"FoldJAX requires Python 3.13; this runtime has "
        f"{sys.version.split()[0]}. In Colab, choose Runtime → Change runtime "
        "type → Runtime version, select a Python 3.13 GPU or TPU runtime, "
        "reconnect, and rerun this notebook from the first cell. If Python "
        "3.13 is not offered, this notebook cannot run in that session."
    )

def detect_accelerator():
    tpu_environment = any(
        os.environ.get(name)
        for name in ("COLAB_TPU_ADDR", "TPU_NAME", "TPU_ACCELERATOR_TYPE")
    )
    if Path("/dev/accel0").exists() or tpu_environment:
        return "tpu"
    if shutil.which("nvidia-smi") is not None:
        gpu_probe = subprocess.run(
            ["nvidia-smi", "-L"],
            check=False,
            capture_output=True,
            text=True,
        )
        if gpu_probe.returncode == 0 and gpu_probe.stdout.strip():
            return "gpu"
    raise RuntimeError(
        "No supported accelerator was found. In Colab select Runtime → Change "
        "runtime type → GPU or TPU, reconnect, and run all cells again."
    )

ACCELERATOR_KIND = detect_accelerator()

def detect_gpu_compute_capability(accelerator_kind):
    if accelerator_kind != "gpu":
        return None
    capability_probe = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=compute_cap",
            "--format=csv,noheader,nounits",
        ],
        check=False,
        capture_output=True,
        text=True,
    )
    if capability_probe.returncode != 0 or not capability_probe.stdout.strip():
        return None
    try:
        major, minor = capability_probe.stdout.splitlines()[0].strip().split(".", 1)
        return int(major), int(minor)
    except (TypeError, ValueError):
        return None

GPU_COMPUTE_CAPABILITY = detect_gpu_compute_capability(ACCELERATOR_KIND)
install_extras = ["cuda12"] if ACCELERATOR_KIND == "gpu" else []
if "alphafold3" in SELECTED_MODELS:
    install_extras.append("alphafold3")
if "openfold3" in SELECTED_MODELS:
    install_extras.append("openfold3-preprocess")
install_profile = "-".join((ACCELERATOR_KIND, *install_extras))
extras_suffix = f"[{','.join(install_extras)}]" if install_extras else ""
foldjax_spec = (
    f"foldjax{extras_suffix} @ "
    f"git+https://github.com/eightmm/FoldJAX.git@{FOLDJAX_REF}"
)
accelerator_requirements = (
    [] if ACCELERATOR_KIND == "gpu" else [f"jax[tpu]=={JAX_VERSION}"]
)
install_marker = Path(
    f"/content/.foldjax-colab-{FOLDJAX_REF}-{install_profile}.installed"
)

if not install_marker.exists():
    foldjax_card(
        "Installing FoldJAX",
        f"Preparing the pinned Python 3.13 {ACCELERATOR_KIND.upper()} stack.",
        metrics=(("source", FOLDJAX_REF[:12]), ("profile", install_profile)),
    )
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--upgrade",
            foldjax_spec,
            *accelerator_requirements,
            "py3Dmol>=2.0,<3",
        ]
    )
    install_marker.write_text(FOLDJAX_REF)
    foldjax_card(
        "Dependencies installed",
        "The runtime will restart once. Reconnect and choose Run all again.",
        tone="success",
    )
    os.kill(os.getpid(), signal.SIGKILL)
else:
    foldjax_card(
        "Dependencies ready",
        "The pinned accelerator stack is already installed in this runtime.",
        tone="success",
        metrics=(("commit", FOLDJAX_REF[:12]), ("profile", install_profile)),
    )


<div style="border:1px solid #dbeafe;border-radius:18px;padding:20px 22px;margin:18px 0 10px;background:linear-gradient(135deg,#eff6ff,#f8fafc)">
  <div style="font-size:12px;font-weight:700;letter-spacing:.12em;color:#2563eb">RUNTIME · STORAGE</div>
  <h3 style="margin:6px 0 4px;color:#0f172a">Fast local execution, independent persistent stores</h3>
  <p style="margin:0;color:#475569">Choose where model assets live while keeping device-specific compilation local.</p>
</div>

Local Colab storage is fastest. Model-asset persistence keeps publisher downloads, converted weights, shared chemistry assets, and generated runtimes across VM replacement. MSA persistence is separate because alignments belong to the input sequence rather than a model. Output persistence keeps prediction trees and resume manifests. Compilation entries always stay local because they are tied to the JAX version, accelerator, options, weights, and concrete input shape.


In [ ]:
# @title 3. Configure storage and portable kernels
# @markdown **Persistence — local runtime storage is fastest**
PERSIST_MODEL_ASSETS_TO_DRIVE = False  # @param {type:"boolean"}
PERSIST_MSA_TO_DRIVE = False  # @param {type:"boolean"}
PERSIST_OUTPUTS_TO_DRIVE = False  # @param {type:"boolean"}
WORK_DIR = Path("/content/foldjax-colab")
foldjax_home = Path("/content/foldjax-cache")
COMPILE_CACHE = WORK_DIR / "compile-cache"

drive_requested = any((
    PERSIST_MODEL_ASSETS_TO_DRIVE,
    PERSIST_MSA_TO_DRIVE,
    PERSIST_OUTPUTS_TO_DRIVE,
))
if drive_requested:
    from google.colab import drive

    drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/FoldJAX")
MODEL_ASSET_STORE = (
    DRIVE_ROOT / "model-assets"
    if PERSIST_MODEL_ASSETS_TO_DRIVE
    else foldjax_home
)
MSA_STORE = (
    DRIVE_ROOT / "msa"
    if PERSIST_MSA_TO_DRIVE
    else foldjax_home / "msa"
)
if PERSIST_OUTPUTS_TO_DRIVE:
    OUTPUT_BASE = DRIVE_ROOT / "outputs"
else:
    OUTPUT_BASE = WORK_DIR / "outputs"

for directory in (
    WORK_DIR, OUTPUT_BASE, COMPILE_CACHE, foldjax_home, MODEL_ASSET_STORE, MSA_STORE
):
    directory.mkdir(parents=True, exist_ok=True)

def connect_store(local_path, target_path, label):
    if local_path == target_path:
        if local_path.is_symlink():
            raise RuntimeError(
                f"{label} still points to Drive from an earlier setting. "
                "Restart the runtime after disabling its Drive persistence."
            )
        local_path.mkdir(parents=True, exist_ok=True)
        return
    target_path.mkdir(parents=True, exist_ok=True)
    if local_path.is_symlink():
        if local_path.resolve() != target_path.resolve():
            raise RuntimeError(
                f"{label} points to a different store. Restart the runtime "
                "after changing its Drive persistence setting."
            )
        return
    if local_path.exists():
        if any(local_path.iterdir()):
            raise RuntimeError(
                f"{label} already has local files. Restart the runtime before "
                "switching it to Drive so no cached data is overwritten."
            )
        local_path.rmdir()
    local_path.symlink_to(target_path, target_is_directory=True)

for component in ("downloads", "weights", "assets", "runtime"):
    connect_store(
        foldjax_home / component,
        MODEL_ASSET_STORE / component,
        f"Model asset directory {component!r}",
    )
connect_store(foldjax_home / "msa", MSA_STORE, "MSA cache")
for model_name in SELECTED_MODELS:
    (foldjax_home / "weights" / model_name).mkdir(parents=True, exist_ok=True)

os.environ["FOLDJAX_HOME"] = str(foldjax_home)
if ACCELERATOR_KIND == "gpu":
    os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.90")
FUSED_TRIANGLE_MIN_CC = (8, 0)
USE_FUSED_TRIANGLE_KERNELS = (
    ACCELERATOR_KIND == "gpu"
    and GPU_COMPUTE_CAPABILITY is not None
    and GPU_COMPUTE_CAPABILITY >= FUSED_TRIANGLE_MIN_CC
)
if USE_FUSED_TRIANGLE_KERNELS:
    triangle_multiplication_backend = "cueq"
    protenix_triangle_backend = "cueq_jit"
    KERNEL_PROFILE = "CUDA fused"
else:
    triangle_multiplication_backend = "xla"
    protenix_triangle_backend = "xla_jit"
    KERNEL_PROFILE = "portable XLA"
os.environ["PROTENIX_TRIANGLE_MULTIPLICATION_BACKEND"] = (
    triangle_multiplication_backend
)
os.environ["PROTENIX_TRIANGLE_BACKEND"] = protenix_triangle_backend
os.environ["BOLTZ_JAX_TRIANGLE_MULTIPLICATION_BACKEND"] = (
    triangle_multiplication_backend
)
foldjax_card(
    "Storage ready",
    "Model assets, MSA results, and outputs follow independent persistence policies.",
    tone="success",
    metrics=(
        ("model assets", MODEL_ASSET_STORE),
        ("MSA", MSA_STORE),
        ("compile cache", COMPILE_CACHE),
        ("Drive root", DRIVE_ROOT if drive_requested else "not mounted"),
        ("model storage", "Drive" if PERSIST_MODEL_ASSETS_TO_DRIVE else "runtime"),
        ("MSA storage", "Drive" if PERSIST_MSA_TO_DRIVE else "runtime"),
        ("output storage", "Drive" if PERSIST_OUTPUTS_TO_DRIVE else "runtime"),
        ("output base", OUTPUT_BASE),
        ("accelerator", ACCELERATOR_KIND.upper()),
        ("kernels", KERNEL_PROFILE),
    ),
)


In [ ]:
# @title 4. Verify Python, JAX, packages, and the selected accelerator
from importlib import metadata

import jax

expected_versions = {
    "cuequivariance": "0.11.1",
    "cuequivariance-jax": "0.11.1",
    "flax": "0.12.9",
    "jaxlib": "0.11.1",
    "qwix": "0.1.8",
    "tokamax": "0.0.13",
}
required_packages = set()
if ACCELERATOR_KIND == "gpu":
    expected_versions.update({
        "cuequivariance-ops-cu12": "0.11.1",
        "cuequivariance-ops-jax-cu12": "0.11.1",
        "jax-cuda12-pjrt": "0.11.1",
        "jax-cuda12-plugin": "0.11.1",
        "triton": "3.7.1",
    })
else:
    required_packages.add("libtpu")
installed_versions = {}
for package in expected_versions:
    try:
        installed_versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed_versions[package] = "missing"
version_mismatches = {
    package: (installed_versions[package], expected)
    for package, expected in expected_versions.items()
    if installed_versions[package] != expected
}
missing_packages = set()
for package in required_packages:
    try:
        metadata.version(package)
    except metadata.PackageNotFoundError:
        missing_packages.add(package)
if jax.__version__ != JAX_VERSION or version_mismatches or missing_packages:
    install_marker.unlink(missing_ok=True)
    raise RuntimeError(
        f"Expected JAX {JAX_VERSION} and the pinned accelerator stack, but got "
        f"JAX {jax.__version__} with mismatches {version_mismatches}. "
        f"Missing required packages: {sorted(missing_packages)}. "
        "Rerun the install cell; it will reinstall the stack and restart once."
    )

all_devices = jax.devices()
accelerator_devices = [
    device for device in all_devices if device.platform == ACCELERATOR_KIND
]
if not accelerator_devices:
    raise RuntimeError(
        f"No JAX {ACCELERATOR_KIND.upper()} was detected after installation. "
        "Reconnect, then run the notebook from the first cell."
    )
device_summary = str(accelerator_devices[0])
if ACCELERATOR_KIND == "gpu":
    gpu_info = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total,driver_version",
            "--format=csv,noheader",
        ],
        check=False,
        capture_output=True,
        text=True,
    )
    if gpu_info.stdout.strip():
        device_summary = gpu_info.stdout.strip().splitlines()[0]
try:
    memory_stats = accelerator_devices[0].memory_stats() or {}
except (AttributeError, RuntimeError, TypeError):
    memory_stats = {}
memory_limit = memory_stats.get("bytes_limit")
DEVICE_MEMORY_BYTES = memory_limit
memory_summary = (
    f"{memory_limit / 2**30:.1f} GiB HBM"
    if memory_limit is not None and ACCELERATOR_KIND == "tpu"
    else f"{memory_limit / 2**30:.1f} GiB VRAM"
    if memory_limit is not None
    else "capacity unavailable from this accelerator runtime"
)
foldjax_card(
    f"{ACCELERATOR_KIND.upper()} runtime verified",
    "Python, JAX, accelerator packages, and selected kernels are consistent.",
    tone="success",
    metrics=(
        ("Python", sys.version.split()[0]),
        ("JAX", jax.__version__),
        ("device", device_summary),
        ("memory", memory_summary),
        ("JAX devices", len(accelerator_devices)),
    ),
)


<div style="border:1px solid #ddd6fe;border-radius:18px;padding:20px 22px;margin:18px 0 10px;background:linear-gradient(135deg,#f5f3ff,#f8fafc)">
  <div style="font-size:12px;font-weight:700;letter-spacing:.12em;color:#7c3aed">CHECKPOINTS · COMPATIBILITY</div>
  <h3 style="margin:6px 0 4px;color:#0f172a">Know what will download before setup begins</h3>
  <p style="margin:0;color:#475569">Selected checkpoints are checked for licence, disk, cache, and runtime requirements before transfer.</p>
</div>

The next cell shows each selected model's publisher, checkpoint licence/terms, registered size, and cache/runtime state before fetching public bundles **sequentially**. AlphaFold3 uses only its managed private parameter directory under the selected model-asset store; upload parameters obtained under DeepMind's terms there. Re-running detects verified assets and skips their download and preparation. OpenFold3 always uses the managed publisher p1 checkpoint. FoldJAX never redistributes restricted model weights.

The disk check is deliberately conservative because source archives and converted JAX files can coexist during first setup. ESMFold2 always uses its released profile in this notebook; because it loads ESMC-6B, the notebook blocks it on T4 and other devices reporting 16 GiB or less.


In [ ]:
# @title 5. Inspect and fetch the selected model weights
import pandas as pd

from foldjax import model_info
MODEL_PROFILES = dict.fromkeys(SELECTED_MODELS)

if "alphafold3" in SELECTED_MODELS:
    foldjax_card(
        "AlphaFold 3 private parameter location",
        "Copy the parameters obtained under DeepMind's terms here. FoldJAX "
        "uses only this managed directory and detects "
        "af3.bin, af3.bin.zst, and supported split parameter files.",
        tone="neutral",
        metrics=(("directory", foldjax_home / "weights" / "alphafold3"),),
    )

model_rows = []
model_infos = {}
model_asset_states = {}
MODEL_WEIGHTS = {}
setup_errors = []
pending_download_bytes = 0
device_memory_bytes = globals().get("DEVICE_MEMORY_BYTES")
for model_name in SELECTED_MODELS:
    info = model_info(model_name)
    model_infos[model_name] = info
    selected_profile = None
    asset_state = {
        "profile": selected_profile,
        "ready": info.weights_ready,
        "download_bytes": info.download_bytes,
        "path": info.weights_path,
        "notes": info.notes,
    }
    model_asset_states[model_name] = asset_state
    MODEL_WEIGHTS[model_name] = None
    if (
        model_name == "esmfold2"
        and selected_profile is None
        and device_memory_bytes is not None
        and device_memory_bytes <= 16 * 2**30
    ):
        setup_errors.append(
            "ESMFold2 Released loads ESMC-6B and is not supported by this "
            "notebook on T4 or other devices reporting 16 GiB or less. "
            "Use a larger-memory runtime or leave ESMFold2 unchecked."
        )
    if not info.weights_fetchable and not asset_state["ready"]:
        setup_errors.append(
            f"{model_name} needs manually obtained weights. Place them at "
            f"{asset_state['path']}. "
            f"{asset_state['notes']}"
        )
    if (
        not asset_state["ready"]
        and asset_state["download_bytes"] is not None
    ):
        pending_download_bytes += asset_state["download_bytes"]
    if model_name == "esmfold2":
        constraint = "complete released bundle ~26.77 GB; >16 GiB device"
    elif model_name == "openfold3":
        constraint = "public managed p1 (~2.29 GB); p2/OpenBind incompatible"
    elif not info.weights_fetchable:
        constraint = "manual/gated parameters"
    else:
        constraint = "public managed bundle"
    model_rows.append(
        {
            "model": info.model,
            "profile": selected_profile or "released",
            "weights": "ready" if asset_state["ready"] else "not ready",
            "runtime_ready": info.runtime.ready,
            "download_GB": (
                round(asset_state["download_bytes"] / 1e9, 2)
                if asset_state["download_bytes"] is not None
                else None
            ),
            "licence": info.weights_licence,
            "constraints": constraint,
            "source": info.weights_source,
        }
    )

foldjax_table(
    pd.DataFrame(model_rows),
    "Selected model compatibility",
    "Review licences, checkpoint size, and cache state before setup.",
)
if setup_errors:
    raise RuntimeError("\n\n".join(setup_errors))
free_bytes = shutil.disk_usage(MODEL_ASSET_STORE).free
required_bytes = max(
    8_000_000_000,
    int(pending_download_bytes * 1.8 + 5_000_000_000),
)
if free_bytes < required_bytes:
    raise RuntimeError(
        f"Model storage {MODEL_ASSET_STORE} has {free_bytes / 1e9:.1f} GB free, "
        f"but this setup requires about {required_bytes / 1e9:.1f} GB for "
        f"{pending_download_bytes / 1e9:.1f} GB of uncached downloads and "
        "conversion headroom. Select fewer models, or select storage with "
        "more free space."
    )
foldjax_card(
    "Storage check passed",
    "Enough room is available for source archives, conversion, and outputs.",
    tone="success",
    metrics=(
        ("free", f"{free_bytes / 1e9:.1f} GB"),
        ("pending downloads", f"{pending_download_bytes / 1e9:.1f} GB"),
        ("setup target", f"{required_bytes / 1e9:.1f} GB"),
    ),
)

for model_name in SELECTED_MODELS:
    info = model_infos[model_name]
    asset_state = model_asset_states[model_name]
    foldjax_card(
        f"Preparing {model_name}",
        "Checkpoint setup runs one model at a time to limit peak storage use.",
        metrics=(("source", info.weights_source),),
    )
    if (
        info.weights_fetchable
        and not asset_state["ready"]
    ):
        fetch_command = [
                sys.executable,
                "-m",
                "foldjax.cli",
                "weights",
                "fetch",
                "--model",
                model_name,
        ]
        subprocess.run(fetch_command, check=True)
    else:
        foldjax_card(
            f"Using cached {model_name} parameters",
            "The verified managed checkpoint is already ready.",
            tone="neutral",
        )
    if not info.runtime.ready:
        foldjax_card(
            f"Building the {model_name} runtime",
            "This one-time generated component is cached with the weights.",
        )
        subprocess.run(
            [
                sys.executable,
                "-m",
                "foldjax.cli",
                "runtime",
                "prepare",
                "--model",
                model_name,
            ],
            check=True,
        )
foldjax_card(
    "All selected models are ready",
    "Checkpoint and generated-runtime setup completed successfully.",
    tone="success",
    metrics=(("models", ", ".join(SELECTED_MODELS)), ("store", foldjax_home)),
)

def directory_inventory(path):
    files = 0
    logical_bytes = 0
    if path.exists():
        for candidate in path.rglob("*"):
            try:
                if candidate.is_file() and not candidate.is_symlink():
                    files += 1
                    logical_bytes += candidate.stat().st_size
            except OSError:
                continue
    return files, logical_bytes

model_location = (
    "Drive" if PERSIST_MODEL_ASSETS_TO_DRIVE else "runtime"
)
msa_location = "Drive" if PERSIST_MSA_TO_DRIVE else "runtime"
output_location = "Drive" if PERSIST_OUTPUTS_TO_DRIVE else "runtime"
cache_specs = (
    ("downloads", foldjax_home / "downloads", model_location, "source reuse"),
    ("weights", foldjax_home / "weights", model_location, "verified reuse"),
    ("shared assets", foldjax_home / "assets", model_location, "verified reuse"),
    ("MSA", foldjax_home / "msa", msa_location, "sequence+provenance"),
    ("generated runtime", foldjax_home / "runtime", model_location, "versioned reuse"),
    ("compilation", COMPILE_CACHE, "runtime", "runtime/device keyed"),
    ("predictions", OUTPUT_BASE, output_location, "manifest resume"),
)
cache_rows = []
for component, path, location, reuse_key in cache_specs:
    file_count, logical_bytes = directory_inventory(path)
    cache_rows.append(
        {
            "component": component,
            "location": location,
            "files": file_count,
            "logical_GB": round(logical_bytes / 1e9, 3),
            "reuse": reuse_key,
            "path": str(path),
        }
    )
foldjax_table(
    pd.DataFrame(cache_rows),
    "Reusable storage inventory",
    "Logical sizes may count hard-linked checkpoint bytes more than once.",
)


<div style="border:1px solid #bbf7d0;border-radius:18px;padding:20px 22px;margin:18px 0 10px;background:linear-gradient(135deg,#ecfdf5,#f8fafc)">
  <div style="font-size:12px;font-weight:700;letter-spacing:.12em;color:#059669">ONE INPUT · MANY MODELS</div>
  <h3 style="margin:6px 0 4px;color:#0f172a">Record the experiment once</h3>
  <p style="margin:0;color:#475569">Protein, nucleic acid, and ligand entities become one reusable common-schema job.</p>
</div>

Form mode uses a colon between protein, DNA, or RNA chains, commas between CCD ligands, and semicolons between SMILES ligands. Upload mode accepts one FoldJAX JSON/YAML job, either from a Colab path or the browser picker, and reuses the cached upload on later Run-all passes. Uploaded jobs unlock common-schema templates, modifications, covalent bonds, affinity requests, and supplied MSA paths without expanding the compact form. Referenced files must also be accessible in the Colab runtime. Every input follows the same compatibility and privacy preflight before any model runs.

MSA policy auto is the default: protein sequences are sent to the public ColabFold MMseqs2 service and the reusable alignment cache is shared across model runs; public MMseqs2 does not serve RNA. Select none when no additional MSA service should be contacted, or required when a missing alignment must stop the run. The notebook prints entity counts, not sequences or SMILES, so a saved output cell does not echo private input text. OpenFold3's compatible p1 checkpoint is a public managed download; AlphaFold3 still requires parameters obtained separately under its publisher's terms. ESMFold2 accepts protein, DNA, RNA, CCD/SMILES ligands, modifications, and covalent bonds; its complete public structure+ESMC+chemistry bundle is about 26.77 GB.


In [ ]:
# @title 6. Configure biomolecules and build the common job
import re

import pandas as pd

from foldjax import Job, model_info
from foldjax.input import _msa_pipeline, _rna_msa_pipeline, msa_search_backend
from foldjax.search import SearchError

# @markdown **Input source**
INPUT_MODE = "Form"  # @param ["Form", "Upload job"]
JOB_FILE_PATH = ""  # @param {type:"string"}
# @markdown **Alignment policy — remote search sends protein sequences to ColabFold**
MSA_POLICY = "auto"  # @param ["auto", "none", "required"]
# @markdown **Form input — use `:` between polymer chains**
JOB_NAME = "protein-rna-atp-demo"  # @param {type:"string"}
PROTEIN_CHAINS = "MKTAYIAKQRQISFVKSHFSRQDILDLWIYHTQGYFPDWQNYTPGPGIRYPLTFGWCFKLVPVDPEEVVEELEKAGVE"  # @param {type:"string"}  # noqa: E501
DNA_CHAINS = ""  # @param {type:"string"}
RNA_CHAINS = "GGGAAACCC"  # @param {type:"string"}
LIGAND_CCD_CODES = "ATP"  # @param {type:"string"}
LIGAND_SMILES = ""  # @param {type:"string"}

NUCLEIC_ALPHABETS = {
    "dna": frozenset("ACGTNRYKMSWBDHV"),
    "rna": frozenset("ACGUNRYKMSWBDHV"),
}

def split_polymer_chains(value, kind):
    compact = re.sub(r"\s+", "", value).upper()
    chains = tuple(part for part in compact.split(":") if part)
    if any(not chain.isalpha() for chain in chains):
        raise ValueError(
            "Polymer chains must contain letters and use ':' between chains."
        )
    allowed = NUCLEIC_ALPHABETS.get(kind)
    if allowed is not None:
        for chain_index, chain in enumerate(chains, start=1):
            invalid = next((base for base in chain if base not in allowed), None)
            if invalid is not None:
                raise ValueError(
                    f"{kind.upper()} chain {chain_index} contains unsupported "
                    f"base {invalid!r}; use IUPAC {''.join(sorted(allowed))}."
                )
    return chains

if INPUT_MODE == "Form":
    protein_sequences = split_polymer_chains(PROTEIN_CHAINS, "protein")
    dna_sequences = split_polymer_chains(DNA_CHAINS, "dna")
    rna_sequences = split_polymer_chains(RNA_CHAINS, "rna")
    ligand_ccds = tuple(
        part.strip().upper()
        for part in LIGAND_CCD_CODES.split(",")
        if part.strip()
    )
    ligand_smiles = tuple(
        part.strip() for part in LIGAND_SMILES.split(";") if part.strip()
    )
    job = Job.from_sequences(
        protein=protein_sequences,
        dna=dna_sequences,
        rna=rna_sequences,
        ligand_ccd=ligand_ccds,
        ligand_smiles=ligand_smiles,
        name=JOB_NAME,
    )
    uploaded_job_path = None
else:
    input_directory = WORK_DIR / "input"
    input_directory.mkdir(parents=True, exist_ok=True)
    upload_marker = input_directory / "last-uploaded-job.txt"
    requested_path = Path(JOB_FILE_PATH).expanduser() if JOB_FILE_PATH.strip() else None
    if requested_path is not None:
        uploaded_job_path = requested_path
    elif upload_marker.is_file() and Path(upload_marker.read_text().strip()).is_file():
        uploaded_job_path = Path(upload_marker.read_text().strip())
    else:
        from google.colab import files

        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError("Upload exactly one FoldJAX JSON or YAML job.")
        uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
        suffix = Path(uploaded_name).suffix.lower()
        if suffix not in {".json", ".yaml", ".yml"}:
            raise ValueError("Uploaded job must use .json, .yaml, or .yml.")
        uploaded_job_path = input_directory / f"uploaded-job{suffix}"
        uploaded_job_path.write_bytes(uploaded_bytes)
        upload_marker.write_text(str(uploaded_job_path))
    if not uploaded_job_path.is_file():
        raise FileNotFoundError(f"Job file does not exist: {uploaded_job_path}")
    job = Job.read(uploaded_job_path)
    JOB_NAME = job.name.strip() or uploaded_job_path.stem

job_document = job.to_document()
job_entities = job_document["entities"]

def entity_copy_count(entity):
    identifiers = entity.get("id")
    return len(identifiers) if isinstance(identifiers, list) else 1

ENTITY_COUNTS = {kind: 0 for kind in ("protein", "dna", "rna", "ligand")}
for entity in job_entities:
    ENTITY_COUNTS[entity["type"]] += entity_copy_count(entity)
PROTEIN_SEQUENCES = tuple(
    entity["sequence"]
    for entity in job_entities
    if entity["type"] == "protein"
    for _copy in range(entity_copy_count(entity))
)
RNA_SEQUENCES = tuple(
    entity["sequence"]
    for entity in job_entities
    if entity["type"] == "rna"
    for _copy in range(entity_copy_count(entity))
)
PROTEIN_MSA_ENTRIES = tuple(
    (entity["sequence"], entity_copy_count(entity), bool(entity.get("unpaired_msa")))
    for entity in job_entities
    if entity["type"] == "protein"
)
RNA_MSA_ENTRIES = tuple(
    (entity["sequence"], entity_copy_count(entity), bool(entity.get("unpaired_msa")))
    for entity in job_entities
    if entity["type"] == "rna"
)
INPUT_ENTITY_TYPES = frozenset(
    kind for kind, count in ENTITY_COUNTS.items() if count
)
if not INPUT_ENTITY_TYPES:
    raise ValueError("Add at least one polymer chain or ligand.")
INPUT_REQUIRED_FEATURES = set()
for entity in job_entities:
    if entity["type"] == "ligand":
        ligand_feature = "ligand_ccd" if entity.get("ccd") else "ligand_smiles"
        INPUT_REQUIRED_FEATURES.add(ligand_feature)
        continue
    for feature in ("unpaired_msa", "paired_msa", "modifications"):
        if entity.get(feature):
            INPUT_REQUIRED_FEATURES.add(feature)
    for template in entity.get("templates", ()):
        mapped = template.get("query_indices") is not None
        INPUT_REQUIRED_FEATURES.add("templates" if mapped else "templates_unmapped")
if MSA_POLICY != "none" and PROTEIN_SEQUENCES:
    INPUT_REQUIRED_FEATURES.add("unpaired_msa")
if job_document.get("bonds"):
    INPUT_REQUIRED_FEATURES.add("bonds")
if job_document.get("properties"):
    INPUT_REQUIRED_FEATURES.add("affinity")
INPUT_REQUIRED_FEATURES = frozenset(INPUT_REQUIRED_FEATURES)

def grouped_sequences(entries):
    groups = {}
    for sequence, copies, supplied in entries:
        key = (sequence, supplied)
        groups[key] = groups.get(key, 0) + copies
    return tuple(
        (sequence, copies, supplied)
        for (sequence, supplied), copies in groups.items()
    )

def verified_msa_cache_state(pipeline, sequence):
    cache_key, _identity = pipeline._identity(sequence)
    directory = pipeline.cache_dir / cache_key
    try:
        cached = pipeline._cached(directory, sequence, cache_key)
    except (SearchError, UnicodeDecodeError, OSError) as error:
        raise RuntimeError(f"Invalid MSA cache entry: {directory}") from error
    return "hit" if cached is not None else "miss"

msa_rows = []
msa_preflight_errors = []
search_backend = msa_search_backend()
protein_search_needed = any(not supplied for _, _, supplied in PROTEIN_MSA_ENTRIES)
protein_pipeline = (
    _msa_pipeline() if MSA_POLICY != "none" and protein_search_needed else None
)
for sequence_index, (sequence, copies, supplied) in enumerate(
    grouped_sequences(PROTEIN_MSA_ENTRIES), start=1
):
    if supplied:
        cache_state = "supplied"
        action = "job alignment; no service contact"
    elif protein_pipeline is None:
        cache_state = "disabled"
        action = "single sequence; no service contact"
    else:
        cache_state = verified_msa_cache_state(protein_pipeline, sequence)
        if cache_state == "hit":
            action = "verified cache; no service contact"
        else:
            backend_kind = search_backend["protein"]["kind"]
            action = f"{backend_kind} search on first model"
    msa_rows.append(
        {
            "entity": f"protein {sequence_index}",
            "length": len(sequence),
            "copies": copies,
            "policy": MSA_POLICY,
            "cache": cache_state,
            "action": action,
        }
    )
rna_search_needed = any(not supplied for _, _, supplied in RNA_MSA_ENTRIES)
rna_pipeline = (
    _rna_msa_pipeline() if MSA_POLICY != "none" and rna_search_needed else None
)
for sequence_index, (sequence, copies, supplied) in enumerate(
    grouped_sequences(RNA_MSA_ENTRIES), start=1
):
    if supplied:
        cache_state = "supplied"
        action = "job alignment; no service contact"
    elif MSA_POLICY == "none":
        cache_state = "disabled"
        action = "single sequence; no service contact"
    elif rna_pipeline is None:
        cache_state = "unavailable"
        action = "no public RNA search; single sequence"
        if MSA_POLICY == "required":
            msa_preflight_errors.append(
                "RNA MSA is required but no local RNA search is configured."
            )
    else:
        cache_state = verified_msa_cache_state(rna_pipeline, sequence)
        action = (
            "verified cache; no service contact"
            if cache_state == "hit"
            else "local RNA search on first model"
        )
    msa_rows.append(
        {
            "entity": f"RNA {sequence_index}",
            "length": len(sequence),
            "copies": copies,
            "policy": MSA_POLICY,
            "cache": cache_state,
            "action": action,
        }
    )
if msa_rows:
    foldjax_table(
        pd.DataFrame(msa_rows),
        "MSA preflight",
        "Identical sequences share one verified search across chains and models.",
    )
if msa_preflight_errors:
    raise RuntimeError("\n".join(msa_preflight_errors))
compatibility_errors = []
for model_name in SELECTED_MODELS:
    capabilities = model_info(model_name).capabilities
    unsupported_types = INPUT_ENTITY_TYPES - set(capabilities.entity_types)
    unsupported_features = INPUT_REQUIRED_FEATURES - set(
        capabilities.common_schema_features
    )
    if unsupported_types or unsupported_features:
        compatibility_errors.append(
            f"{model_name} cannot represent this common input: "
            f"entity types={sorted(unsupported_types)}, "
            f"features={sorted(unsupported_features)}"
        )
if compatibility_errors:
    raise RuntimeError("\n\n".join(compatibility_errors))

JOB_SLUG = re.sub(r"[^a-z0-9._-]+", "-", JOB_NAME.lower()).strip("-")
if not JOB_SLUG:
    raise ValueError("JOB_NAME must contain at least one letter or digit.")
entity_summary = ", ".join(
    f"{kind}={count}" for kind, count in ENTITY_COUNTS.items() if count
)

job_path = (
    job.write(WORK_DIR / "input" / f"{JOB_SLUG}.json")
    if uploaded_job_path is None
    else uploaded_job_path
)
foldjax_card(
    "Common job recorded",
    "Every selected backend will read this same validated common input file.",
    tone="success",
    metrics=(
        ("job", JOB_SLUG),
        ("entities", entity_summary),
        ("input file", job_path),
    ),
)


<div style="border:1px solid #fed7aa;border-radius:18px;padding:20px 22px;margin:18px 0 10px;background:linear-gradient(135deg,#fff7ed,#f8fafc)">
  <div style="font-size:12px;font-weight:700;letter-spacing:.12em;color:#ea580c">EXECUTION · SEQUENTIAL ACCELERATOR SESSIONS</div>
  <h3 style="margin:6px 0 4px;color:#0f172a">Review the plan, then run safely</h3>
  <p style="margin:0;color:#475569">Each backend gets the same recorded job while retaining its own validated runtime options.</p>
</div>

Every selected model receives the same common job path, seed policy, MSA policy, depth cap, and sampling mode. **Default** leaves samples, diffusion steps, and recycles unset so every backend uses its own native defaults; **Fast demo** and **Custom** apply an explicit shared schedule. The plan also resolves representation support and trunk/full stopping before inference. Pair representations can be very large because their storage grows quadratically with token count. Setup-only mode completes installation, checkpoint, input, MSA, compatibility, and request validation without launching inference. FoldJAX releases each backend session before opening the next, while resume mode avoids repeating completed work.

If **Continue on error** is enabled, an out-of-memory or model-specific setup failure is recorded while later models still run. Programming defects and user cancellation still stop immediately.


In [ ]:
# @title 7. Configure and review every model run
from foldjax import PredictionRequest, model_info, resolve_requests

# @markdown **Run policy**
RUN_MODE = "Default"  # @param ["Fast demo", "Default", "Custom"]
EXECUTION = "Predict"  # @param ["Predict", "Setup only"]
CONTINUE_ON_ERROR = True  # @param {type:"boolean"}
# @markdown **Shared custom schedule — applies to every selected model; select one model for individual tuning**
CUSTOM_NUM_SAMPLES = 1  # @param {type:"integer", min:1, max:10}
CUSTOM_NUM_STEPS = 100  # @param {type:"integer", min:1, max:1000}
CUSTOM_NUM_RECYCLES = 3  # @param {type:"integer", min:1, max:20}
# @markdown **Seeds — explicit values replace the generated `0..NUM_SEEDS-1` sequence**
NUM_SEEDS = 1  # @param {type:"integer", min:1, max:5}
EXPLICIT_SEEDS = ""  # @param {type:"string"}
# @markdown **Optional memory override — applies to every run mode**
MAX_MSA_DEPTH = 0  # @param {type:"integer", min:0, max:4096}
# @markdown **Advanced output — optional**
REPRESENTATIONS = "None"  # @param ["None", "Single", "Single + pair", "All available"]
STOP_AFTER = "Full structure"  # @param ["Full structure", "Trunk representations"]

PREFLIGHT_ONLY = EXECUTION == "Setup only"
if RUN_MODE == "Custom" and min(
    int(CUSTOM_NUM_SAMPLES), int(CUSTOM_NUM_STEPS), int(CUSTOM_NUM_RECYCLES)
) < 1:
    raise ValueError("Custom samples, steps, and recycles must be positive.")
seed_text = EXPLICIT_SEEDS.strip()
if seed_text:
    try:
        RESOLVED_EXPLICIT_SEEDS = tuple(
            int(part.strip()) for part in seed_text.split(",") if part.strip()
        )
    except ValueError as error:
        raise ValueError("EXPLICIT_SEEDS must be comma-separated integers.") from error
    if not RESOLVED_EXPLICIT_SEEDS or any(seed < 0 for seed in RESOLVED_EXPLICIT_SEEDS):
        raise ValueError("Explicit seeds must be non-negative integers.")
    if len(set(RESOLVED_EXPLICIT_SEEDS)) != len(RESOLVED_EXPLICIT_SEEDS):
        raise ValueError("Explicit seeds must be unique.")
    if len(RESOLVED_EXPLICIT_SEEDS) > 10:
        raise ValueError("Use at most 10 explicit seeds in the Colab workflow.")
    EFFECTIVE_NUM_SEEDS = len(RESOLVED_EXPLICIT_SEEDS)
    SEED_SOURCE = "explicit values"
else:
    if not 1 <= int(NUM_SEEDS) <= 5:
        raise ValueError("NUM_SEEDS must be between 1 and 5.")
    RESOLVED_EXPLICIT_SEEDS = None
    EFFECTIVE_NUM_SEEDS = int(NUM_SEEDS)
    SEED_SOURCE = f"generated 0..{EFFECTIVE_NUM_SEEDS - 1}"
if int(MAX_MSA_DEPTH) < 0:
    raise ValueError("MAX_MSA_DEPTH must be 0 or a positive row cap.")
if STOP_AFTER == "Trunk representations" and REPRESENTATIONS == "None":
    raise ValueError("Trunk-only execution requires saved representations.")
if RUN_MODE == "Fast demo":
    RUN_LABEL = "fast-demo"
    SAMPLING_OVERRIDE = {"num_samples": 1, "num_steps": 20, "num_recycles": 1}
elif RUN_MODE == "Custom":
    RUN_LABEL = (
        f"custom-s{int(CUSTOM_NUM_SAMPLES)}-n{int(CUSTOM_NUM_STEPS)}-"
        f"r{int(CUSTOM_NUM_RECYCLES)}"
    )
    SAMPLING_OVERRIDE = {
        "num_samples": int(CUSTOM_NUM_SAMPLES),
        "num_steps": int(CUSTOM_NUM_STEPS),
        "num_recycles": int(CUSTOM_NUM_RECYCLES),
    }
elif RUN_MODE == "Default":
    RUN_LABEL = "default"
    SAMPLING_OVERRIDE = {}
else:
    raise ValueError(f"Unknown RUN_MODE: {RUN_MODE}")
OUTPUT_ROOT = OUTPUT_BASE / JOB_SLUG / RUN_LABEL
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

triangle_kernel = "cueq" if USE_FUSED_TRIANGLE_KERNELS else "xla"
alphafold_attention = "auto" if ACCELERATOR_KIND == "gpu" else "xla"
MODEL_PRECISION = {
    "protenix": "native bfloat16",
    "opendde": "float32",
    "boltz2": "native bfloat16",
    "esmfold2": "native mixed/bfloat16",
    "openfold3": "float32",
    "alphafold3": "native bfloat16",
}
MODEL_OPTIONS = {
    "protenix": {
        "attention_kernel": "xla",
        "triangle_kernel": triangle_kernel,
    },
    "opendde": {"dtype": "float32", "attention_kernel": "xla"},
    "boltz2": {
        "attention_kernel": "xla",
        "triangle_kernel": triangle_kernel,
    },
    "esmfold2": {},
    "openfold3": {"triangle_kernel": triangle_kernel},
    "alphafold3": {"attention_kernel": alphafold_attention},
}

def representation_request(model_name):
    available = tuple(model_info(model_name).capabilities.representations)
    requested = {
        "None": (),
        "Single": ("single",),
        "Single + pair": ("single", "pair"),
        "All available": available,
    }[REPRESENTATIONS]
    missing = set(requested) - set(available)
    if missing or (REPRESENTATIONS == "All available" and not available):
        raise ValueError(
            f"{model_name} cannot provide {REPRESENTATIONS}: "
            f"available={available or 'none'}"
        )
    return requested or None

REPRESENTATION_REQUESTS = {
    model_name: representation_request(model_name) for model_name in SELECTED_MODELS
}
SEED_ARGUMENTS = (
    {"seeds": RESOLVED_EXPLICIT_SEEDS}
    if RESOLVED_EXPLICIT_SEEDS is not None
    else {"num_seeds": EFFECTIVE_NUM_SEEDS}
)
STOP_POINT = "trunk" if STOP_AFTER == "Trunk representations" else "full"
model_requests = tuple(
    PredictionRequest(
        model=model_name,
        input=job_path,
        weights=MODEL_WEIGHTS[model_name],
        output_dir=OUTPUT_ROOT / model_name / JOB_SLUG,
        cache_dir=COMPILE_CACHE,
        seed=0,
        max_msa_depth=int(MAX_MSA_DEPTH) or None,
        msa=MSA_POLICY,
        options=MODEL_OPTIONS[model_name],
        profile=MODEL_PROFILES[model_name],
        representations=REPRESENTATION_REQUESTS[model_name],
        stop_after=STOP_POINT,
        resume=True,
        on_error="continue" if CONTINUE_ON_ERROR else "stop",
        **SEED_ARGUMENTS,
        **SAMPLING_OVERRIDE,
    )
    for model_name in SELECTED_MODELS
)
resolved_runs = tuple(
    resolve_requests(model_request)[0] for model_request in model_requests
)
plan_rows = [
    {
        "model": run.model,
        "input": Path(run.input).name,
        "output": str(run.output_dir),
        "seeds": list(run.resolved_seeds),
        "seed count": EFFECTIVE_NUM_SEEDS,
        "seed source": SEED_SOURCE,
        "profile": run.profile or "released",
        "MSA cap": run.max_msa_depth or "model default",
        "representations": run.representations or "none",
        "stop": run.stop_after,
        "options": run.options or "model runtime defaults",
        "precision": MODEL_PRECISION[run.model],
        "schedule": run.sampling or "native default (varies by model)",
    }
    for run in resolved_runs
]
foldjax_table(
    pd.DataFrame(plan_rows),
    "Resolved execution plan",
    "Each row is independently validated but uses the same common input.",
)


In [ ]:
# @title 8. Run all selected models
import json

from foldjax import BatchReport, predict_batch, progress

partial_reports = []
if not PREFLIGHT_ONLY:
    progress.enable()
    for model_request in model_requests:
        foldjax_card(
            f"Running {model_request.model}",
            "The accelerator session opens for this model and closes before the next.",
            metrics=(
                ("seeds", model_request.resolved_seeds),
                ("MSA", model_request.msa),
            ),
        )
        partial_reports.append(predict_batch(model_request))
report = BatchReport(
    results=tuple(
        result for partial in partial_reports for result in partial.results
    ),
    failures=tuple(
        failure for partial in partial_reports for failure in partial.failures
    ),
    skipped=tuple(
        path for partial in partial_reports for path in partial.skipped
    ),
)
if PREFLIGHT_ONLY:
    foldjax_card(
        "Setup-only complete — inference skipped",
        "Installation, weights, input, MSA, compatibility, and requests are ready.",
        tone="success",
        metrics=(("planned models", len(model_requests)), ("predictions", 0)),
    )
else:
    foldjax_card(
        "Prediction batch complete",
        "Successful outputs are ready for validation and side-by-side analysis.",
        tone="warning" if report.failures else "success",
        metrics=(
            ("results", len(report.results)),
            ("failures", len(report.failures)),
            ("resumed", len(report.skipped)),
        ),
    )
if report.failures:
    foldjax_table(
        pd.DataFrame(failure.summary() for failure in report.failures),
        "Model failures",
        "These runs failed; later models continued because Continue on error "
        "is enabled.",
    )


<div style="border:1px solid #bae6fd;border-radius:18px;padding:20px 22px;margin:18px 0 10px;background:linear-gradient(135deg,#f0f9ff,#f8fafc)">
  <div style="font-size:12px;font-weight:700;letter-spacing:.12em;color:#0284c7">ANALYSIS · NATIVE SCORES</div>
  <h3 style="margin:6px 0 4px;color:#0f172a">Compare evidence, not a fabricated leaderboard</h3>
  <p style="margin:0;color:#475569">Structures are parsed and scores validated before side-by-side inspection.</p>
</div>

The table keeps each model's native score names. A similarly named metric can still differ in calibration, input representation, and sampling schedule, so the notebook does not combine them into a universal ranking. Use the table to inspect consistency and uncertainty, and use the interactive viewer for structural comparison.


In [ ]:
# @title 9. Validate structures and build the comparison table
import math

import gemmi

comparison_rows = []
STRUCTURES = {}
for result in report.results:
    for sample_index, sample in enumerate(result.samples, start=1):
        if sample.structure_path is None:
            raise RuntimeError(f"{result.model} returned no structure path.")
        structure_path = Path(sample.structure_path)
        if not structure_path.is_file() or structure_path.stat().st_size == 0:
            raise RuntimeError(f"Missing or empty structure: {structure_path}")
        if structure_path.suffix.lower() in {".cif", ".mmcif"}:
            gemmi.cif.read_file(str(structure_path))
        else:
            gemmi.read_structure(str(structure_path))
        if not sample.scores or not all(
            math.isfinite(float(value)) for value in sample.scores.values()
        ):
            raise RuntimeError(
                f"{result.model} returned invalid confidence scores: "
                f"{sample.scores}"
            )

        label = f"{result.model} · seed {sample.seed} · sample {sample_index}"
        STRUCTURES[label] = structure_path
        row = {
            "model": result.model,
            "seed": sample.seed,
            "sample": sample_index,
            "structure": structure_path.name,
        }
        row.update(
            {
                f"score:{name}": float(value)
                for name, value in sorted(sample.scores.items())
            }
        )
        comparison_rows.append(row)

if comparison_rows:
    comparison = pd.DataFrame(comparison_rows).sort_values(
        ["model", "seed", "sample"],
        ignore_index=True,
    )
    comparison_title = "Validated prediction comparison"
    comparison_message = (
        "Native confidence fields stay model-specific; no artificial global rank "
        "is added."
    )
elif PREFLIGHT_ONLY:
    comparison = pd.DataFrame(plan_rows)
    comparison_title = "Validated setup-only plan"
    comparison_message = "No model inference was launched."
elif STOP_POINT == "trunk":
    representation_rows = []
    for result in report.results:
        if result.representations is None:
            raise RuntimeError(f"{result.model} returned no representations.")
        summary = result.representations.summary()
        representation_rows.append(
            {
                "model": result.model,
                "representations": ", ".join(summary["names"]),
                "archive": summary["path"],
            }
        )
    if not representation_rows:
        raise RuntimeError("No trunk representation run completed.")
    comparison = pd.DataFrame(representation_rows)
    comparison_title = "Trunk representation outputs"
    comparison_message = "Trunk-only mode intentionally produces no structures."
else:
    raise RuntimeError(
        "No prediction produced a usable structure. Review the failure table above."
    )
foldjax_table(comparison.fillna("—"), comparison_title, comparison_message)
foldjax_card(
    "Analysis ready",
    "The requested prediction or setup artifacts passed validation.",
    tone="success",
    metrics=(
        ("models", comparison["model"].nunique()),
        ("structures", len(STRUCTURES)),
    ),
)


<div style="border:1px solid #a7f3d0;border-radius:18px;padding:20px 22px;margin:18px 0 10px;background:linear-gradient(135deg,#ecfeff,#f8fafc)">
  <div style="font-size:12px;font-weight:700;letter-spacing:.12em;color:#0f766e">VISUALIZATION · STRUCTURE EXPLORER</div>
  <h3 style="margin:6px 0 4px;color:#0f172a">Inspect every model, seed, and sample interactively</h3>
  <p style="margin:0;color:#475569">Switch predictions and coloring without rerunning any model.</p>
</div>

Select any model/sample result without rerunning prediction. Chain spectrum is best for complexes; confidence coloring reads the structure's B-factor field when the writer stores per-residue confidence there.


In [ ]:
# @title 10. Interactive model and color selector
import ipywidgets as widgets
import py3Dmol
from IPython.display import clear_output

prediction_selector = (
    widgets.Dropdown(
        options=list(STRUCTURES),
        description="Prediction:",
        layout=widgets.Layout(width="70%"),
    )
    if STRUCTURES
    else None
)
color_selector = (
    widgets.Dropdown(
        options=("Chain spectrum", "Confidence (B-factor)"),
        description="Color:",
        layout=widgets.Layout(width="45%"),
    )
    if STRUCTURES
    else None
)
viewer_output = widgets.Output() if STRUCTURES else None


def render_structure(*_changes):
    structure_path = STRUCTURES[prediction_selector.value]
    structure_format = (
        "cif"
        if structure_path.suffix.lower() in {".cif", ".mmcif"}
        else "pdb"
    )
    with viewer_output:
        clear_output(wait=True)
        viewer = py3Dmol.view(width=900, height=560)
        viewer.addModel(structure_path.read_text(), structure_format)
        if color_selector.value == "Confidence (B-factor)":
            viewer.setStyle(
                {
                    "cartoon": {
                        "colorscheme": {
                            "prop": "b",
                            "gradient": "roygb",
                            "min": 0,
                            "max": 100,
                        }
                    }
                }
            )
        else:
            viewer.setStyle({"cartoon": {"color": "spectrum"}})
        viewer.setBackgroundColor("white")
        viewer.zoomTo()
        viewer.show()


if STRUCTURES:
    foldjax_card(
        "Structure explorer ready",
        "Switch model, seed, sample, and coloring without rerunning.",
        tone="success",
        metrics=(("predictions", len(STRUCTURES)), ("viewer", "py3Dmol")),
    )
    prediction_selector.observe(render_structure, names="value")
    color_selector.observe(render_structure, names="value")
    display(widgets.VBox([prediction_selector, color_selector, viewer_output]))
    render_structure()
else:
    foldjax_card(
        "Structure explorer skipped",
        "Setup-only and trunk-only modes do not produce structures.",
        tone="neutral",
        metrics=(("predictions", 0),),
    )


<div style="border:1px solid #cbd5e1;border-radius:18px;padding:20px 22px;margin:18px 0 10px;background:linear-gradient(135deg,#f8fafc,#f1f5f9)">
  <div style="font-size:12px;font-weight:700;letter-spacing:.12em;color:#475569">EXPORT · REPRODUCIBLE BUNDLE</div>
  <h3 style="margin:6px 0 4px;color:#0f172a">Keep the input, analysis, and artifacts together</h3>
  <p style="margin:0;color:#475569">Create one portable archive for review, sharing, or later reproduction.</p>
</div>

The archive contains the common input job, a CSV plan/comparison table, the full batch report, and every produced model artifact. Full runs include structures and confidence files; trunk runs include representation archives; setup-only runs produce a compact validated plan without inference artifacts.


In [ ]:
# @title 11. Package all model outputs
import zipfile

DOWNLOAD_RESULTS = True  # @param {type:"boolean"}

comparison_path = WORK_DIR / f"{JOB_SLUG}-comparison.csv"
report_path = WORK_DIR / f"{JOB_SLUG}-batch-report.json"
archive_path = WORK_DIR / f"{JOB_SLUG}-foldjax-comparison.zip"
comparison.to_csv(comparison_path, index=False)
report_path.write_text(json.dumps(report.summary(), indent=2))

artifact_count = 0
with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(job_path, f"input/job{job_path.suffix.lower()}")
    bundle.write(comparison_path, "comparison.csv")
    bundle.write(report_path, "batch_report.json")
    for artifact in sorted(OUTPUT_ROOT.rglob("*")):
        if artifact.is_file():
            artifact_count += 1
            bundle.write(
                artifact,
                Path("outputs") / artifact.relative_to(OUTPUT_ROOT),
            )

if PREFLIGHT_ONLY:
    archive_kind = "setup"
elif STOP_POINT == "trunk":
    archive_kind = "trunk"
else:
    archive_kind = "prediction"
foldjax_card(
    f"{archive_kind.title()} archive ready",
    "The bundle contains the common job, validated table, report, and any "
    "produced model artifacts.",
    tone="success",
    metrics=(
        ("file", archive_path.name),
        ("size", f"{archive_path.stat().st_size / 1e6:.1f} MB"),
        ("artifacts", artifact_count + 3),
    ),
)
if DOWNLOAD_RESULTS:
    try:
        from google.colab import files

        files.download(str(archive_path))
    except ImportError:
        foldjax_card(
            "Automatic download unavailable",
            "Download the archive from the notebook file browser.",
            tone="warning",
        )


<div style="border:1px solid #e2e8f0;border-radius:18px;padding:20px 22px;margin:18px 0 10px;background:linear-gradient(135deg,#ffffff,#f8fafc)">
  <div style="font-size:12px;font-weight:700;letter-spacing:.12em;color:#64748b">NEXT · CHOOSE THE RIGHT SCHEDULE</div>
  <h3 style="margin:6px 0 4px;color:#0f172a">Use model defaults unless you need a faster smoke test</h3>
  <p style="margin:0;color:#475569">Default preserves every backend's native schedule; Fast demo is the lower-cost workflow check.</p>
</div>

- Keep **Default** for each backend's native sample, diffusion-step, and recycle settings. Use **Fast demo** for a lower-cost smoke test. **Custom** applies one explicitly recorded schedule to every selected model; select one model when tuning it individually.
- Add more seeds to inspect within-model variability. NUM_SEEDS generates `0..N-1`; a comma-separated explicit seed list replaces that generated sequence, and the resolved plan shows the effective count and source.
- Use the MSA depth cap when alignments exceed accelerator memory. Zero preserves each model's released/default cap.
- Use **Setup only** to validate installation, checkpoints, input, MSA state, model compatibility, and resolved requests without inference.
- Start with Protenix only if storage is tight. OpenDDE and Boltz-2 add public all-atom comparisons. ESMFold2 accepts the same all-biomolecule common job, but its complete structure+ESMC+chemistry bundle is about 26.77 GB.
- OpenFold3 downloads or reuses the managed public p1 checkpoint. Upstream p2/OpenBind checkpoints are not compatible with this port. AlphaFold3 still needs a supplied parameter directory.
- Re-running the prediction cell uses resume manifests. Turn on output persistence to keep those prediction trees across VM replacement. A different input, model list, shape, options, weights, or runtime identity gets separate validated work.
- Protein MSA entries are keyed by sequence and search provenance, shared across selected models, and reused without contacting the service again. The MSA preflight shows hit or miss without printing the sequence. Model-asset and MSA persistence are independent: retain model assets for repeated model use, and retain MSA only when the same sequences will be revisited.
- Form mode covers compact protein, DNA, RNA, CCD, and SMILES input. Upload a common JSON/YAML job for templates, modifications, covalent bonds, affinity, supplied MSAs, or reusable experiments.
- Representation controls are advanced: pair arrays are quadratic, and trunk-only runs produce no structures.
